# Index a spectral database using Flash Entropy 

Input from MSP file and export to a pickle file of indexed database, for fast search.

Pickle files of unknown origin is a big security risk. Both MSP and pickle formats can be compressed to much smaller files.

In [1]:
# use asari 1.17 or later; install ms_entropy if needed
# !pip3 install --upgrade asari-metabolomics

In [2]:
# import pickle
# import ms_entropy as ME
from asari.annotate import *

In [3]:
MSP_dict = { # can add more here for controlled volcabulary
    'MW': 'ExactMass',
    'PRECURSORMZ': 'precursor_mz',
}

def msp_standarize(LL, MSP_dict):
    for e in LL:
        for k,v in MSP_dict.items():
            if k in e and v not in e:
                e[v] = float(e[k])
        e['Num Peaks'] = int(e['Num Peaks'])
        if 'RETENTIONTIME' in e and e['RETENTIONTIME']:
            e['RETENTIONTIME'] = float(e['RETENTIONTIME'])
        else: 
            e['RETENTIONTIME'] = None
        e['PRECURSORMZ'] = float(e['PRECURSORMZ'])
        if 'COLLISIONENERGY' in e:
            try:
                e['COLLISIONENERGY'] = float(e['COLLISIONENERGY'])
            except ValueError:
                e['COLLISIONENERGY'] = None

    return LL

In [4]:
msp_file = "/Users/sli/Downloads/MSMS-Public_experimentspectra-pos-VS19.msp"
db = msp_standarize(parse_msp_to_listdict(msp_file), MSP_dict)

In [5]:
db[99]

{'NAME': 'Theobromine; CE0; YAPQBXQYLJRXSA-UHFFFAOYSA-N',
 'PRECURSORMZ': 181.072006225586,
 'PRECURSORTYPE': '[M+H]+',
 'FORMULA': 'C7H8N4O2',
 'Ontology': 'Xanthines',
 'INCHIKEY': 'YAPQBXQYLJRXSA-UHFFFAOYSA-N',
 'SMILES': 'Cn1cnc2c1c(nc(=O)n2C)O',
 'RETENTIONTIME': 1.692147,
 'CCS': '137.8588614',
 'IONMODE': 'Positive',
 'COLLISIONENERGY': 0.0,
 'Comment': 'MS2 deconvoluted using MS2Dec from all ion fragmentation data, MetaboLights identifier MTBLS1040; YAPQBXQYLJRXSA-UHFFFAOYSA-N_STSL_0032_Theobromine_8000fmol_180416_S2_LC02_MS02_45',
 'Num Peaks': 3,
 'peaks': [(138.0668, 1.0), (181.07249, 100.0), (182.07561, 8.0)],
 'precursor_mz': 181.072006225586}

In [6]:
entropy = ME.FlashEntropySearch()
entropy.build_index(db)
pickle.dump(entropy, open("indexed_MSMS-Public_experimentspectra-pos-VS19.pkl", 'wb'))

# Summary

Some MSP files may have inconsistent formats. One can modify the msp_standarize function to fix those. 

Input can also be JSON if one loads JSON directly and run it through FlashEntropySearch indexing. 